# 🚦 Indian Traffic Sign Learning Assistant — CNN Training v2

## Why Transfer Learning?
The dataset has only ~82 **unique** images per class (the rest are augmented duplicates).  
Training a CNN from scratch requires thousands of unique images per class.  
**MobileNetV2** (pre-trained on ImageNet) solves this — it already knows edges, textures, shapes.  
We just teach it the last layer: *which Indian traffic sign is this?*

### ✅ Expected Results
- **Phase 1** (frozen backbone): ~50–70% val accuracy in 10 epochs  
- **Phase 2** (fine-tune): ~80–95% val accuracy in 10 more epochs  
- Total time on T4 GPU: **~8–12 minutes**

> 💡 **Before starting:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — GPU Check & Constants                             ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, tensorflow as tf, numpy as np
from PIL import Image
import os, json, zipfile, csv, time
from pathlib import Path

print('=' * 60)
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  NO GPU — go to Runtime → Change runtime type → T4 GPU!')

print(f'✅ TensorFlow {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'✅ TF sees GPU: {gpus[0].name}')
print('=' * 60)

# ── Hyperparameters ───────────────────────────────────────────
IMG_SIZE    = 96     # MobileNetV2 works well at 96–224
BATCH_SIZE  = 64
PHASE1_LR   = 1e-3   # frozen backbone
PHASE2_LR   = 5e-5   # fine-tune — must be small!
PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 20
VAL_SPLIT   = 0.15
SEED        = 42

print(f'\n📐 IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}')
print(f'   Phase-1: {PHASE1_EPOCHS} epochs @ lr={PHASE1_LR}')
print(f'   Phase-2: {PHASE2_EPOCHS} epochs @ lr={PHASE2_LR}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Upload Dataset ZIP                                ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import files

print('📁 Upload your dataset zip (archive (5).zip)...')
uploaded   = files.upload()
zip_fname  = list(uploaded.keys())[0]
ZIP_PATH   = f'/content/{zip_fname}'

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'\n✅ {zip_fname}  ({size_mb:.1f} MB)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Extract & Build Smart Train/Val Split             ║
# ║  KEY FIX: split by SOURCE image so augmented copies of the  ║
# ║  same photo never appear in both train AND val.             ║
# ╚══════════════════════════════════════════════════════════════╝
import zipfile, csv, re
from pathlib import Path

EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('📦 Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

entries = list(Path(EXTRACT_DIR).iterdir())
DATASET_ROOT = str(entries[0]) if len(entries) == 1 and entries[0].is_dir() else EXTRACT_DIR
IMAGES_ROOT  = Path(DATASET_ROOT) / 'Images'
print(f'✅ Extracted → {DATASET_ROOT}')

# Read class map
CLASS_MAP = {}
with open(Path(DATASET_ROOT) / 'traffic_sign.csv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        CLASS_MAP[int(row['ClassId'].strip())] = row['Name'].strip()

SORTED_IDS  = sorted(CLASS_MAP.keys())
ID_TO_IDX   = {cid: idx for idx, cid in enumerate(SORTED_IDS)}
NUM_CLASSES = len(SORTED_IDS)
IDX_TO_NAME = {str(idx): CLASS_MAP[cid] for idx, cid in enumerate(SORTED_IDS)}
print(f'✅ {NUM_CLASSES} classes loaded')

# ── Source-aware split ────────────────────────────────────────
# Extract the 'root source' UUID from filenames like:
#   0_original_43_original_43.png_UUID1_UUID2.png
# We group by class_id + first UUID → same source

def get_source_key(fname: str, class_id: int) -> str:
    """Return a key identifying the original source image."""
    uuids = re.findall(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', fname)
    if uuids:
        return f'{class_id}_{uuids[0]}'   # group by first UUID = original image
    return f'{class_id}_{fname}'           # no UUID → it IS an original

# Collect all images grouped by source
from collections import defaultdict
source_to_files = defaultdict(list)  # source_key → [(img_path, label_idx)]
total = 0

for cid in SORTED_IDS:
    class_dir = IMAGES_ROOT / str(cid)
    if not class_dir.exists(): continue
    for img_path in class_dir.glob('*.png'):
        key = get_source_key(img_path.name, cid)
        source_to_files[key].append((str(img_path), ID_TO_IDX[cid]))
        total += 1

# Split sources into train/val
rng        = np.random.default_rng(SEED)
all_keys   = list(source_to_files.keys())
rng.shuffle(all_keys)
split_pt   = int(len(all_keys) * (1 - VAL_SPLIT))
train_keys = set(all_keys[:split_pt])
val_keys   = set(all_keys[split_pt:])

train_files = [(p, l) for k in train_keys for p, l in source_to_files[k]]
val_files   = [(p, l) for k in val_keys   for p, l in source_to_files[k]]

print(f'\n📊 Source-aware split:')
print(f'   Unique sources  : {len(all_keys):,}')
print(f'   Train files     : {len(train_files):,}  ({len(train_keys):,} sources)')
print(f'   Val files       : {len(val_files):,}  ({len(val_keys):,} sources)')
print(f'   Total images    : {total:,}')
print(f'\n✅ NO source leakage between train and val!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Build tf.data Pipelines (load from paths)         ║
# ╚══════════════════════════════════════════════════════════════╝

# Augmentation (NO horizontal flip — it changes sign meaning!)
augment_layer = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomBrightness(0.20),
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
], name='augmentation')

# MobileNetV2 preprocessing
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input  # scales to [-1, 1]

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    img = preprocess(img)
    return img, label

def load_and_augment(path, label):
    img, label = load_and_preprocess(path, label)
    # Undo preprocess, augment, re-preprocess
    img = (img + 1.0) * 127.5                          # back to [0, 255]
    img = augment_layer(img, training=True)
    img = preprocess(img)                               # back to [-1, 1]
    return img, label

def make_dataset(file_label_pairs, augment=False, shuffle=True):
    paths  = [p for p, _ in file_label_pairs]
    labels = [l for _, l in file_label_pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=4096, seed=SEED)
    fn = load_and_augment if augment else load_and_preprocess
    ds = ds.map(fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_files, augment=True,  shuffle=True)
val_ds   = make_dataset(val_files,   augment=False, shuffle=False)

print(f'✅ Pipelines ready')
print(f'   Train batches: {len(train_ds)}  |  Val batches: {len(val_ds)}')

# Sanity check a batch
for imgs, lbls in train_ds.take(1):
    print(f'   Batch shape  : {imgs.shape}  labels: {lbls.shape}')
    print(f'   Pixel range  : [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Build Transfer Learning Model (MobileNetV2)       ║
# ╚══════════════════════════════════════════════════════════════╝

from tensorflow.keras import layers, models

# ── Frozen backbone ───────────────────────────────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'      # pre-trained on 1.2M ImageNet images!
)
base_model.trainable = False   # freeze all backbone weights

# ── Custom classification head ────────────────────────────────
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)    # training=False keeps BN in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name='TrafficSign_MobileNetV2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

total_params     = model.count_params()
trainable_params = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'✅ Model built')
print(f'   Total params     : {total_params:,}')
print(f'   Trainable params : {trainable_params:,}  (head only — backbone frozen)')
print(f'   Frozen params    : {total_params - trainable_params:,}  (MobileNetV2 backbone)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Phase 1: Train Head (backbone frozen)             ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt

print('🚀 PHASE 1: Training classification head (backbone frozen)...')
print(f'   Learning rate: {PHASE1_LR}  |  Max epochs: {PHASE1_EPOCHS}')
print()

p1_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2,
        min_lr=1e-6, verbose=1
    ),
]

t0 = time.time()
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS, callbacks=p1_callbacks
)
p1_time = time.time() - t0

p1_best_acc  = max(history1.history['val_accuracy']) * 100
p1_epochs    = len(history1.history['loss'])
print(f'\n✅ Phase 1 done in {p1_time/60:.1f} min')
print(f'   Best val accuracy: {p1_best_acc:.2f}%  ({p1_epochs} epochs)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Phase 2: Fine-Tune Top Layers of Backbone         ║
# ╚══════════════════════════════════════════════════════════════╝

# Unfreeze the top 40 layers of MobileNetV2 (keep lower layers frozen)
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_now = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'🔓 Unfroze top 40 backbone layers')
print(f'   Now trainable: {trainable_now:,} params')

# MUST recompile with a much smaller LR after unfreezing
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'\n🚀 PHASE 2: Fine-tuning  (lr={PHASE2_LR})...')
print()

p2_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.4, patience=2,
        min_lr=1e-8, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    )
]

t0 = time.time()
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE2_EPOCHS, callbacks=p2_callbacks
)
p2_time = time.time() - t0

p2_best_acc  = max(history2.history['val_accuracy']) * 100
p2_epochs    = len(history2.history['loss'])
total_time   = (p1_time + p2_time) / 60

print(f'\n✅ Phase 2 done in {p2_time/60:.1f} min')
print(f'   Best val accuracy: {p2_best_acc:.2f}%  ({p2_epochs} epochs)')
print(f'\n🎉 TOTAL training time: {total_time:.1f} minutes')
print(f'   Final accuracy: {p2_best_acc:.2f}%')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Plot Training Curves                              ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0d1117')

def style_ax(ax, title):
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.xaxis.label.set_color('#8b949e')
    ax.yaxis.label.set_color('#8b949e')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold')
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

# Combine both phases
p1_len   = len(history1.history['accuracy'])
p2_len   = len(history2.history['accuracy'])
ep1      = range(1, p1_len + 1)
ep2      = range(p1_len + 1, p1_len + p2_len + 1)

for ax, metric, ylabel in [
    (axes[0], 'accuracy', 'Accuracy'),
    (axes[1], 'loss',     'Loss')
]:
    style_ax(ax, ylabel)
    ax.plot(ep1, history1.history[metric],         color='#f97316', linewidth=2, label='Train P1')
    ax.plot(ep1, history1.history[f'val_{metric}'],color='#fb923c', linewidth=2, linestyle='--', label='Val P1')
    ax.plot(ep2, history2.history[metric],         color='#3b82f6', linewidth=2, label='Train P2')
    ax.plot(ep2, history2.history[f'val_{metric}'],color='#60a5fa', linewidth=2, linestyle='--', label='Val P2')
    ax.axvline(x=p1_len + 0.5, color='#f8c471', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(p1_len + 0.7, ax.get_ylim()[1] * 0.95, 'Fine-tune →',
            color='#f8c471', fontsize=9)
    ax.set_xlabel('Epoch')
    ax.legend(facecolor='#0d1117', labelcolor='white', fontsize=9)
    if metric == 'accuracy':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))

plt.suptitle(f'🚦 MobileNetV2 Transfer Learning — Best Val Accuracy: {p2_best_acc:.1f}%',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Save Model & Class Map                            ║
# ╚══════════════════════════════════════════════════════════════╝
from sklearn.metrics import classification_report

# ── Final evaluation ──────────────────────────────────────────
print('📊 Final evaluation...')
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'   Val Loss     : {val_loss:.4f}')
print(f'   Val Accuracy : {val_acc*100:.2f}%')

# Per-class report
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([y for _, y in val_ds], axis=0)
class_names = [IDX_TO_NAME[str(i)] for i in range(NUM_CLASSES)]
print('\n' + classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# ── Save model in Keras native format ─────────────────────────
MODEL_PATH    = '/content/traffic_model.keras'
MODEL_H5_PATH = '/content/traffic_model.h5'
CLASS_MAP_PATH = '/content/class_map.json'
SUMMARY_PATH   = '/content/training_summary.json'

model.save(MODEL_PATH)       # native Keras format
model.save(MODEL_H5_PATH)    # legacy HDF5 (for compatibility)
print(f'\n✅ Model saved:')
print(f'   {MODEL_PATH}    ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')
print(f'   {MODEL_H5_PATH} ({os.path.getsize(MODEL_H5_PATH)/1e6:.1f} MB)')

with open(CLASS_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(IDX_TO_NAME, f, indent=2, ensure_ascii=False)
print(f'✅ class_map.json saved')

# Save metadata so app.py knows which model type to load
summary = {
    'architecture':      'MobileNetV2_transfer_learning',
    'num_classes':       NUM_CLASSES,
    'image_size':        IMG_SIZE,
    'total_images':      len(train_files) + len(val_files),
    'unique_sources':    len(all_keys),
    'best_val_accuracy': round(float(p2_best_acc), 2),
    'best_val_loss':     round(float(val_loss), 4),
    'phase1_epochs':     p1_epochs,
    'phase2_epochs':     p2_epochs,
    'preprocessing':     'mobilenet_v2',  # app.py must use this!
}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ training_summary.json saved')
print(f'\n🎉 Done! Run Cell 10 to download.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Download Everything                              ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import files

print('⬇️  Downloading files (browser will prompt for each)...')
print()

for fpath, dest in [
    ('/content/traffic_model.keras',     'models/traffic_model.keras  ← primary'),
    ('/content/traffic_model.h5',        'models/traffic_model.h5     ← backup'),
    ('/content/class_map.json',          'models/class_map.json'),
    ('/content/training_summary.json',   'models/training_summary.json'),
    ('/content/training_curves.png',     'anywhere'),
]:
    print(f'  {os.path.basename(fpath):30s} → place in {dest}')
    files.download(fpath)

print()
print('=' * 60)
print('✅ NEXT STEPS ON YOUR LOCAL MACHINE:')
print()
print('  1. Create the models/ folder if it does not exist')
print('  2. Place traffic_model.keras + class_map.json inside it')
print('  3. Run the app:')
print()
print('     .\\venv\\Scripts\\python.exe app/app.py')
print()
print('  4. Open http://localhost:7860  🚀')
print('=' * 60)